# E v6: extractive NQ and independent FEVER NLI

28 reviewed development samples plus the previous 100 development regression samples.
The two cohorts overlap on 5 IDs; do not combine them as 128 independent samples.
FEVER uses a pinned MNLI classifier, not Qwen. Its cost and accuracy are reported separately.
NQ uses the pinned Qwen NF4 model and selects exact evidence spans once.
All paired semantic review remains pending. This is development, not held-out evaluation.


In [ ]:
import json, os, sys, subprocess, hashlib, tarfile, shutil
from pathlib import Path
import torch
assert sys.version_info >= (3, 11), '需要 Python 3.11+'
assert torch.cuda.is_available(), '请选择 GPU 运行时'
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
EXPECTED_SHA256 = '74d40b01c96245e1caf2ac8af786bf3b05394971863bddc81a75a2a2afa45ef5'
BUNDLE_NAME = 'answer_repair_v6r1_reviewed_gpu'
OUTPUT = WORK / (BUNDLE_NAME.removesuffix('_gpu') + '_outputs')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0), 'CUDA:', torch.version.cuda)


In [ ]:
# Kaggle may unpack the uploaded archive before attaching it.
EXPECTED_INDEX_SHA256 = '1c095e9fb52077898e8eb85672f76ba4ac052e87b9e099da61df523be404d0bd'
archive = WORK / (BUNDLE_NAME + '.tar.gz')
extract_root = WORK / ('e_reviewed_' + EXPECTED_SHA256[:12])
BUNDLE = extract_root / BUNDLE_NAME
transport = 'archive_sha256'
if not archive.exists() and Path('/kaggle/input').exists():
    candidates = list(Path('/kaggle/input').rglob(archive.name))
    if candidates:
        assert len(candidates) == 1, '实验归档必须唯一'
        archive = candidates[0]
    else:
        indexes = [p for p in Path('/kaggle/input').rglob('bundle_checksums.json')
                   if p.parent.name == BUNDLE_NAME]
        assert len(indexes) == 1, '请附加唯一的冻结实验包输入'
        assert hashlib.sha256(indexes[0].read_bytes()).hexdigest() == EXPECTED_INDEX_SHA256
        source = indexes[0].parent
        source_checksums = json.loads(indexes[0].read_text())
        assert all(hashlib.sha256((source / p).read_bytes()).hexdigest() == h
                   for p,h in source_checksums.items()), '已解压输入文件哈希不一致'
        if not BUNDLE.exists():
            shutil.copytree(source, BUNDLE)
        transport = 'pinned_checksum_index_and_all_files'
elif not archive.exists():
    from google.colab import files
    uploaded = files.upload()
    assert archive.name in uploaded, '请选择配套实验包'
    archive.write_bytes(uploaded[archive.name])
if transport == 'archive_sha256':
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == EXPECTED_SHA256
    if not BUNDLE.exists():
        extract_root.mkdir(parents=True, exist_ok=True)
        with tarfile.open(archive) as handle:
            handle.extractall(extract_root, filter='data')
assert hashlib.sha256((BUNDLE / 'bundle_checksums.json').read_bytes()).hexdigest() == EXPECTED_INDEX_SHA256
checksums = json.loads((BUNDLE / 'bundle_checksums.json').read_text())
assert all(hashlib.sha256((BUNDLE / p).read_bytes()).hexdigest() == h for p,h in checksums.items())
manifest = json.loads((BUNDLE / 'manifest.json').read_text())
integrity = {'transport': transport, 'expected_archive_sha256': EXPECTED_SHA256,
             'checksum_index_sha256': EXPECTED_INDEX_SHA256,
             'verified_files': len(checksums), 'all_files_verified': True}
(OUTPUT / 'input_integrity.json').write_text(json.dumps(integrity, indent=2))
print('输入校验通过:', integrity)
print({ds: r['groups'] for ds,r in manifest['datasets'].items()})


In [ ]:
def run_logged(argv, name):
    with (OUTPUT / name).open('a', encoding='utf-8') as log:
        log.write('COMMAND ' + json.dumps(argv) + '\n'); log.flush()
        process = subprocess.Popen(argv, cwd=BUNDLE, env={**os.environ, 'PYTHONUNBUFFERED': '1'},
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
    if status: raise subprocess.CalledProcessError(status, argv)

run_logged([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-answer-repair.txt'], 'install.log')
run_logged([sys.executable, 'scripts/check_answer_repair_environment.py', '--require-cuda',
            '--out', str(OUTPUT / 'environment.json')], 'environment.log')
run_logged([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests'], 'tests.log')
for path in ('manifest.json', 'bundle_checksums.json'):
    shutil.copy2(BUNDLE / path, OUTPUT / path)
RUNS = []
for ds in ('fever', 'nq'):
    for arm in manifest['protocol']:
        if arm.get('dataset', ds) != ds: continue
        strategy = arm['strategy']
        result = OUTPUT / f'{ds}_{strategy}.jsonl'
        argv = [sys.executable, '-u', 'scripts/run_answer_repair.py', '--dataset', ds,
                '--c1', str(BUNDLE / f'inputs/{ds}_reviewed_c1.jsonl'), '--all',
                '--strategy', strategy, '--max-new-tokens', str(arm['max_new_tokens']), '--out', str(result)]
        RUNS.append((ds, strategy, result, argv))
        run_logged(argv + ['--check-only'], f'{ds}_{strategy}_preflight.log')


## Real GPU experiments
Run both dataset-specific arms, then the fixed regression cohort.


In [ ]:
for ds, strategy, result, argv in RUNS:
    print('开始:', ds, strategy, flush=True)
    run_logged(argv, f'{ds}_{strategy}_run.log')


In [ ]:
for ds, strategy, result, argv in RUNS:
    run_logged([sys.executable, 'scripts/summarize_reviewed_answer_repair.py',
                '--manifest', str(BUNDLE / 'manifest.json'),
                '--c1', str(BUNDLE / f'inputs/{ds}_reviewed_c1.jsonl'),
                '--results', str(result), '--out-prefix', str(OUTPUT / f'{ds}_{strategy}_reviewed')],
               f'{ds}_{strategy}_summary.log')
# 将无复核与有复核两组的候选作比较，避免把候选生成差异归因于复核。
comparison = {}
for ds in ('fever', 'nq') if any(a['strategy']=='guarded_reanswer' for a in manifest['protocol']) else ():
    load = lambda strategy: [json.loads(l) for l in (OUTPUT / f'{ds}_{strategy}.jsonl').read_text().splitlines()]
    plain, guarded = load('query_grounded'), load('guarded_reanswer')
    assert [r['sample_id'] for r in plain] == [r['sample_id'] for r in guarded]
    comparison[ds] = {'candidate_output_mismatch_ids': [a['sample_id'] for a,b in zip(plain,guarded)
        if a['raw_output'] != b['raw_output'] or a['validation_input'] != b['validation_input']]}
(OUTPUT / 'candidate_comparison.json').write_text(json.dumps(comparison, ensure_ascii=False, indent=2))
print('候选一致性检查:', comparison)
print('人工成对审核表已生成，所有结果审核栏仍为空。')


In [ ]:
# Fixed previous development cohort; additional regression checks, no tuning.
for arm in manifest['protocol']:
    ds, strategy = arm['dataset'], arm['strategy']
    result = OUTPUT / f'{ds}_{strategy}_dev50.jsonl'
    argv = [sys.executable, '-u', 'scripts/run_answer_repair.py', '--dataset', ds,
            '--c1', str(BUNDLE / f'inputs/{ds}_dev50_c1.jsonl'), '--all',
            '--strategy', strategy, '--max-new-tokens', str(arm['max_new_tokens']), '--out', str(result)]
    run_logged(argv + ['--check-only'], f'{ds}_{strategy}_dev50_preflight.log')
    run_logged(argv, f'{ds}_{strategy}_dev50_run.log')


In [ ]:
export = shutil.make_archive(str(WORK / (OUTPUT.name + '_export')), 'zip', OUTPUT)
print('结果包:', export)
print('SHA256:', hashlib.sha256(Path(export).read_bytes()).hexdigest())
if not Path('/kaggle/working').exists():
    from google.colab import files
    files.download(export)
